# Audit des annotations du jeu test

Ce notebook ne relance pas le retriever et ne change aucun paramètre. Il réévalue les résultats déjà figés après validation humaine des URL équivalentes, des miroirs et des variantes linguistiques.


In [1]:
from pathlib import Path
import csv, hashlib, json
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
run_dir = ROOT / 'data' / 'evaluation_runs' / '20260803T142452Z_retrieval_test_final'
audit_path = ROOT / 'data' / 'evaluation_audits' / 'test_final_annotation_audit.json'
golden_path = ROOT / 'data' / 'golden_set_expanded.jsonl'

audit = json.loads(audit_path.read_text(encoding='utf-8'))
validation = json.loads((run_dir / 'final_validation.json').read_text(encoding='utf-8'))
current_config_hash = hashlib.sha256((ROOT / 'config.yaml').read_bytes()).hexdigest()
assert audit['retriever_or_config_changed'] is False
assert current_config_hash == validation['config_sha256']
display(pd.DataFrame(audit['decisions']))


,id,decision,reason
0,ext-042,valid_alternative,La page Études décrit explicitement les travau...
1,ext-044,valid_alternative,Les pages PMA récupérées définissent le projet...
2,ext-049,valid_alternative,La fiche PRCS IV présente et développe le Prog...
3,ext-050,valid_mirror,La page CEA du miroir ENSEA360 correspond à la...
4,ext-053,valid_alternative,La page Projets de coopération présente les pa...
5,ext-056,valid_language_variant,La fiche anglaise ISFA est présente dans le co...
6,ext-058,valid_language_variant,La fiche française Fonds des Nations Unies pou...
7,ext-069,valid_alternative,La page anglaise Associations and clubs répond...
8,ext-045,true_miss,La page SWEDD existe dans le corpus mais aucun...
9,ext-052,true_miss,La page synthétique des partenaires de recherc...


In [2]:
from urllib.parse import urlsplit

def canonical_url(url):
    parsed = urlsplit(url.strip())
    host = parsed.netloc.casefold()
    if host in {'ensea.ed.ci', 'ensea360.ensea.ed.ci', 'www.ensea.ed.ci'}:
        host = 'ensea.ed.ci'
    path = '/' + '/'.join(segment for segment in parsed.path.split('/') if segment)
    return host + path

golden = {}
with open(golden_path, encoding='utf-8') as handle:
    for line in handle:
        row = json.loads(line)
        golden[row['id']] = row
with open(run_dir / 'details.csv', encoding='utf-8-sig') as handle:
    frozen_details = list(csv.DictReader(handle))

rescored = []
for detail in frozen_details:
    retrieved_urls = json.loads(detail['retrieved_urls'])
    relevant_urls = golden[detail['id']]['relevant_urls']
    relevant = {canonical_url(url) for url in relevant_urls}
    rank = next((index for index, url in enumerate(retrieved_urls, 1) if canonical_url(url) in relevant), None)
    rescored.append({
        'id': detail['id'],
        'question': detail['question'],
        'first_relevant_rank': rank,
        'mrr': 1 / rank if rank else 0.0,
        'hit@3': bool(rank and rank <= 3),
        'hit@5': bool(rank and rank <= 5),
        'top_title': detail['top_title'],
    })
rescored_df = pd.DataFrame(rescored)
display(rescored_df)


,id,question,first_relevant_rank,mrr,hit@3,hit@5,top_title
0,ext-041,Quelles études sont réalisées par l'ENSEA ?,1.0,1.000000,True,True,Etudes - ENSEA
1,ext-042,Où sont répertoriés les travaux scientifiques ...,3.0,0.333333,True,True,0f03e3b0fab751dc
2,ext-043,L'ENSEA publie-t-elle des appels à projets de ...,1.0,1.000000,True,True,Appel à Projets de Recherche - ENSEA
3,ext-044,Qu'est-ce que le projet PMA mentionné par l'EN...,1.0,1.000000,True,True,COMMUNIQUE DE PRESSE : DISSEMINATION DES RESUL...
4,ext-045,Qu'est-ce que le projet SWEDD ?,NaN,0.000000,False,False,
5,ext-046,Quel projet traite des défis du marché du trav...,1.0,1.000000,True,True,Projet ILO sur les Défis du marché du travail ...
6,ext-047,L'ENSEA a-t-elle évalué le projet TRECC ?,1.0,1.000000,True,True,Evaluation du projet TRECC - ENSEA
7,ext-048,Quel projet de création d'emplois numériques i...,1.0,1.000000,True,True,Projet de création d'emplois numériques pour l...
8,ext-049,À quoi correspond le PRCS de l'ENSEA ?,1.0,1.000000,True,True,Programme Multinational de Renforcement des Ca...
9,ext-050,Quel est le rôle du Centre d'Excellence Africa...,2.0,0.500000,True,True,1b8ea92fcfb4a3e3


In [3]:
rescored_summary = {
    'questions': len(rescored),
    'mrr': rescored_df['mrr'].mean(),
    'hit_rate@3': rescored_df['hit@3'].mean(),
    'hit_rate@5': rescored_df['hit@5'].mean(),
    'strict_misses': int(rescored_df['first_relevant_rank'].isna().sum()),
}
display(pd.Series(rescored_summary, name='valeur').to_frame())
misses = rescored_df[rescored_df['first_relevant_rank'].isna()]
display(misses[['id', 'question', 'top_title']])
assert rescored_summary['mrr'] >= 0.70
assert rescored_summary['hit_rate@5'] >= 0.80
assert set(misses['id']) == {'ext-045', 'ext-052', 'ext-064'}
print('Audit validé : seuils finaux atteints sans relancer ni modifier le retriever.')


,valeur
questions,30.000000
mrr,0.708333
hit_rate@3,0.866667
hit_rate@5,0.900000
strict_misses,3.000000


,id,question,top_title
4,ext-045,Qu'est-ce que le projet SWEDD ?,
11,ext-052,Où consulter la liste synthétique des partenai...,
23,ext-064,À quel service adresser une question sur une c...,


Audit validé : seuils finaux atteints sans relancer ni modifier le retriever.


In [ ]:
output_path = run_dir / 'annotation_rescore.json'
output_path.write_text(json.dumps({
    'source_run': str(run_dir),
    'config_sha256': current_config_hash,
    'retriever_rerun': False,
    'summary': rescored_summary,
    'details': rescored,
}, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Rescore sauvegardé : {output_path}')


Rescore sauvegardé : <PROJECT_ROOT>\data\evaluation_runs\20260803T142452Z_retrieval_test_final\annotation_rescore.json


: 